In [5]:
# =========================================================
# STEP 1: Data Loading, Encoding, and Splitting (7-Day)
# =========================================================
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

data_path = '../data/final_model_ready_pune_data_7day.csv'
print("Loading 7-day model ready data...")
df = pd.read_csv(data_path)
df['arrival_date'] = pd.to_datetime(df['arrival_date'])

# 1. Encode Categoricals for LightGBM
categorical_cols = ['mandi_name', 'district', 'state', 'variety']
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')

# 2. Chronological Train/Test Split
# Using Nov 1st, 2024 as the split. Everything after is unseen test data.
split_date = '2024-11-01' 
train_df = df[df['arrival_date'] < split_date].copy()
test_df = df[df['arrival_date'] >= split_date].copy()

# 3. Drop Leaky Columns (Keeping the safe weather lags!)
drop_cols = [
    'arrival_date', 'target_price', 'commodity', 
    'modal_price', 'min_price', 'max_price'
]

X_train = train_df.drop(columns=drop_cols, errors='ignore')
y_train = train_df['target_price']

X_test = test_df.drop(columns=drop_cols, errors='ignore')
y_test = test_df['target_price']

print("\n--- CHECKPOINT 1 COMPLETE ---")
print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape:  {X_test.shape}")
print(f"Features used: {list(X_train.columns)}")

Loading 7-day model ready data...

--- CHECKPOINT 1 COMPLETE ---
Training features shape: (17893, 28)
Testing features shape:  (6918, 28)
Features used: ['mandi_name', 'district', 'state', 'variety', 'is_real_trade', 'day_of_week', 'month', 'day_of_year', 'is_weekend', 'is_holiday', 'price_lag_7', 'price_lag_8', 'price_lag_9', 'price_lag_14', 'price_lag_30', 'price_roll_mean_7', 'price_roll_std_7', 'price_roll_mean_30', 'price_expanding_mean', 'sin_365_1', 'cos_365_1', 'sin_365_2', 'cos_365_2', 'temp_mean_lag7', 'rainfall_lag7', 'rainfall_7d_sum', 'rainfall_30d_sum', 'temp_7d_avg']


In [8]:
# =========================================================
# STEP 2: LightGBM Datasets & Quantile Training (7-Day)
# =========================================================
import lightgbm as lgb
import os

print("Creating LightGBM datasets...")

# 1. Create Base Datasets
lgb_train = lgb.Dataset(X_train, label=y_train, categorical_feature=categorical_cols, free_raw_data=False)
lgb_eval = lgb.Dataset(X_test, label=y_test, categorical_feature=categorical_cols, reference=lgb_train, free_raw_data=False)

# 2. Define Base Parameters and Quantile-Specific Tuning
base_params = {
    'boosting_type': 'gbdt',
    'learning_rate': 0.01,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'random_state': 42
}

quantile_configs = {
    'p50': {
        'alpha': 0.50, 'max_depth': 10, 'num_leaves': 63, 
        'min_data_in_leaf': 15, 'lambda_l1': 0.1, 'lambda_l2': 0.1
    },
    'p10': {
        'alpha': 0.05, 'max_depth': 6, 'num_leaves': 31, 
        'min_data_in_leaf': 30, 'lambda_l1': 1.0, 'lambda_l2': 1.0
    },
    'p90': {
        'alpha': 0.95, 'max_depth': 6, 'num_leaves': 31, 
        'min_data_in_leaf': 30, 'lambda_l1': 1.0, 'lambda_l2': 1.0
    }
}

models = {}
os.makedirs('../models', exist_ok=True)
print("\nStarting 7-Day Horizon Probabilistic Training...")

# 3. Train the 3 Models
for name, config in quantile_configs.items():
    print(f"\n--- Training {name} Model (Alpha={config['alpha']}) ---")
    
    # Merge base params with specific config
    params = {**base_params, **config}
    params['objective'] = 'quantile'
    params['metric'] = 'quantile'

    callbacks = [
        lgb.early_stopping(stopping_rounds=100, first_metric_only=False),
        lgb.log_evaluation(period=200)
    ]

    model = lgb.train(
        params,
        lgb_train,
        num_boost_round=3000,
        valid_sets=[lgb_train, lgb_eval],
        valid_names=['train', 'eval'],
        callbacks=callbacks
    )
    
    models[name] = model
    
    # Save the model
    model_path = f"../models/lightgbm_onion_7day_{name}.txt"
    model.save_model(model_path)
    print(f"Saved: {model_path}")

print("\n--- CHECKPOINT 2 COMPLETE: All 7-Day Models Trained & Saved ---")

Creating LightGBM datasets...

Starting 7-Day Horizon Probabilistic Training...

--- Training p50 Model (Alpha=0.5) ---
Training until validation scores don't improve for 100 rounds
[200]	train's quantile: 93.6387	eval's quantile: 201.493
[400]	train's quantile: 70.5483	eval's quantile: 189.244
[600]	train's quantile: 63.944	eval's quantile: 187.441
Early stopping, best iteration is:
[516]	train's quantile: 66.2708	eval's quantile: 186.8
Saved: ../models/lightgbm_onion_7day_p50.txt

--- Training p10 Model (Alpha=0.05) ---
Training until validation scores don't improve for 100 rounds
[200]	train's quantile: 31.7563	eval's quantile: 42.2391
[400]	train's quantile: 25.3541	eval's quantile: 39.8039
[600]	train's quantile: 22.2497	eval's quantile: 38.9874
Early stopping, best iteration is:
[571]	train's quantile: 22.5208	eval's quantile: 38.9237
Saved: ../models/lightgbm_onion_7day_p10.txt

--- Training p90 Model (Alpha=0.95) ---
Training until validation scores don't improve for 100 rounds

In [9]:
# =========================================================
# STEP 3: Generate Formal Performance Metrics (7-Day)
# =========================================================
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error

print("Generating 7-Day predictions on unseen test data...\n")

# Create a results dataframe to hold true values and predictions
results_df = test_df[['arrival_date', 'mandi_name', 'target_price']].copy()

# Generate predictions for all 3 quantiles
for name in ['p10', 'p50', 'p90']:
    results_df[f'{name}_pred'] = models[name].predict(X_test)

# 1. Standard Regression Metrics (Using p50 Median Forecast)
mae = mean_absolute_error(results_df['target_price'], results_df['p50_pred'])
rmse = np.sqrt(mean_squared_error(results_df['target_price'], results_df['p50_pred']))
mape = mean_absolute_percentage_error(results_df['target_price'], results_df['p50_pred'])

# 2. Probabilistic Metrics (Using p10 and p90 Bounds)
results_df['in_bound'] = (results_df['target_price'] >= results_df['p10_pred']) & (results_df['target_price'] <= results_df['p90_pred'])
coverage = results_df['in_bound'].mean() * 100

print("="*50)
print("FINAL 7-DAY HORIZON METRICS FOR YOUR PAPER")
print("="*50)
print(f"MAE:  ₹ {mae:.2f}")
print(f"RMSE: ₹ {rmse:.2f}")
print(f"MAPE: {mape*100:.2f}%")
print(f"80% Prediction Interval Coverage: {coverage:.1f}%")
print("="*50)

Generating 7-Day predictions on unseen test data...

FINAL 7-DAY HORIZON METRICS FOR YOUR PAPER
MAE:  ₹ 373.60
RMSE: ₹ 652.66
MAPE: 19.45%
80% Prediction Interval Coverage: 79.7%
